# W5C1: From ingredients to a search engine

Run every cell from the top. **Everything already works.**

We run this together, cell by cell. Each part ends with a **TRY IT**: one question, one empty cell. After the break you get a pantry and 25 minutes.

Today you will:

1. Watch **Ctrl+F** fail at a question it cannot answer.
2. Turn a recipe into a **row of numbers**: tf.
3. Weight it by **idf**, how rare each ingredient is.
4. Rank by the **angle** between rows, and cook dinner.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first.
import numpy as np
import pandas as pd

recipes = pd.read_csv("data/recipes.csv")

print(recipes.shape[0], "recipes")
print()
print(recipes.head(4).to_string(index=False))


---

## First, the problem

You have **chickpeas, spinach, yoghurt, cumin, garlic and lemon**. What can you
cook?

Try it the obvious way. `.str.contains` is exactly what Ctrl+F does: look for
this run of characters inside that text.


In [ ]:
PANTRY = "chickpeas spinach yoghurt cumin garlic lemon"

print("Ctrl+F for the whole list:",
      int(recipes["ingredients"].str.contains(PANTRY).sum()), "recipes")
print()
print("Ctrl+F for one ingredient at a time:")
for ingredient in PANTRY.split():
    hits = int(recipes["ingredients"].str.contains(ingredient).sum())
    print(f"   {ingredient:10s} {hits:3d} recipes")

uses_everything = recipes["ingredients"].apply(
    lambda text: all(word in text for word in PANTRY.split()))
print()
print("recipes using all six:", int(uses_everything.sum()))


Zero for the whole list, 42 for `garlic`, and **no recipe uses all six**.

There is nothing to find here. There is only something to **rank**: which recipe
uses the most of what you have, counting the rare ingredients for more than the
ones every recipe has. That is what the rest of today builds.

<img src="images/ctrl-f.png" width="640">


## Part 1. tf: how much of this recipe is that ingredient?


Every recipe becomes a **row of numbers**, one slot per ingredient in a shared
vocabulary. That is what `CountVectorizer` does, written out so you can see it.

tf fills the slots: how often the word appears here, divided by the recipe's
length so long recipes do not win by being long.


<img src="images/tf.png" width="640">

In [ ]:
# Four tiny recipes, so every number fits on the screen.
DOCS = [
    "chickpeas lemon parsley olive oil salt",
    "chickpeas tahini lemon cumin olive oil salt",
    "flour butter sugar eggs salt",
    "flour butter sugar olive oil salt",
]

for number, document in enumerate(DOCS, 1):
    print(f"recipe {number}: {document}")

In [ ]:
# The vocabulary: every distinct word, in a fixed order. Slot i is the same
# ingredient in every row, which is the only reason two rows can be compared.
VOCAB = []
for document in DOCS:
    for word in document.split():
        if word not in VOCAB:
            VOCAB.append(word)
VOCAB = sorted(VOCAB)

print(len(VOCAB), "distinct ingredients")
print(VOCAB)

In [ ]:
# Count the words of one recipe into an array, then divide by its length.
def term_frequencies(document):
    """One row of tf values, one slot per word in VOCAB."""
    words = document.split()
    counts = np.zeros(len(VOCAB))
    for word in words:
        if word in VOCAB:              # a pantry may hold things we have never seen
            counts[VOCAB.index(word)] = counts[VOCAB.index(word)] + 1
    return counts / len(words)


print("recipe 1:", DOCS[0])
print()
print(pd.Series(term_frequencies(DOCS[0]), index=VOCAB).round(3).to_string())
print()
print("Every ingredient scores 0.167. tf rates salt exactly as highly as")
print("chickpeas, which is the problem Part 2 fixes.")

In [ ]:
# ================== TRY IT 1 ==================
# Salt is in all four recipes. Is its tf the same in all four?
# ==============================================


## Part 2. idf: how much does the ingredient narrow it down?


Salt is in every recipe, so knowing a dish contains salt tells you nothing.

Weight each word by how rare it is. Count the recipes containing it, its
**document frequency**, and take the log of the total over that. A word in every
recipe scores exactly zero.


<img src="images/idf.png" width="640">

In [ ]:
# One count per vocabulary word: how many recipes contain it at all.
document_frequency = np.zeros(len(VOCAB))
for document in DOCS:
    for word in set(document.split()):
        document_frequency[VOCAB.index(word)] = document_frequency[VOCAB.index(word)] + 1

# np.log works on the whole array at once. No loop.
IDF = np.log(len(DOCS) / document_frequency)

print(pd.DataFrame({"df": document_frequency.astype(int), "idf": IDF.round(3)},
                   index=VOCAB).to_string())
print()
print("salt is in all four, so log(4/4) = 0 and it is switched off. Not by a")
print("stop-word list: by arithmetic.")

In [ ]:
# tf-idf is the two arrays multiplied slot by slot: (12,) * (12,) -> (12,).
def tf_idf(document):
    """One row of tf-idf weights, one slot per word in VOCAB."""
    return term_frequencies(document) * IDF


print("recipe 1, every ingredient weighted:")
print(pd.Series(tf_idf(DOCS[0]), index=VOCAB).round(4).to_string())

In [ ]:
# ================== TRY IT 2 ==================
# Which ingredient carries the most weight in recipe 1?
# What weight does salt get?
# ==============================================


## Part 3. Cosine similarity: compare directions, not lengths


Every recipe is a row of 12 numbers, so comparing two recipes means comparing two
vectors.

Distance is the wrong tool: a nine-ingredient stew and a three-ingredient one sit
far apart because one has more numbers in it. **Cosine** ignores length and
measures the angle. Same direction 1, nothing in common 0.


<img src="images/cosine-similarity.png" width="640">

In [ ]:
# Cosine similarity: a dot product over two lengths. NumPy has all three.
def cosine(first, second):
    """The cosine of the angle between two tf-idf rows."""
    lengths = np.linalg.norm(first) * np.linalg.norm(second)
    if lengths == 0:
        return 0.0
    return np.dot(first, second) / lengths


# Stack the four rows into one array: 4 recipes x 12 ingredients.
all_weights = np.array([tf_idf(document) for document in DOCS])
print("all_weights.shape:", all_weights.shape)
print()

scores = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        scores[i][j] = cosine(all_weights[i], all_weights[j])

labels = ["recipe 1", "recipe 2", "recipe 3", "recipe 4"]
print(pd.DataFrame(scores.round(3), index=labels, columns=labels).to_string())
print()
print("The two chickpea recipes score 0.289 against each other and exactly 0.000")
print("against the baking. Nothing told it about cuisine; salt cancelled out and")
print("what was left was chickpeas.")

In [ ]:
# A PANTRY is just another short recipe. Weight it the same way and compare.
pantry = "chickpeas lemon"

pantry_weights = tf_idf(pantry)

print("pantry:", pantry)
print()
for number in range(4):
    print(f"  recipe {number + 1}  {cosine(pantry_weights, all_weights[number]):.3f}   {DOCS[number]}")
print()
print("That is a search engine. Everything after this is the same idea, faster.")

In [ ]:
# ================== TRY IT 3 ==================
# Search for `olive oil salt` instead. Would you trust the ranking it
# gives you?
# ==============================================


## Part 4. One hundred recipes


Back to the real corpus and the pantry Ctrl+F could not help with.

`TfidfVectorizer` does Parts 1 to 3 in two lines, over all hundred at once.


<img src="images/activity-pantry.png" width="640">

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Only the INGREDIENTS are indexed. You search ingredients and get back a name,
# so the answer is never something you could have searched for directly.
vectorizer = TfidfVectorizer()
recipe_vectors = vectorizer.fit_transform(recipes["ingredients"])

print("recipe_vectors.shape:", recipe_vectors.shape)
print("   ", recipe_vectors.shape[0], "recipes x",
      recipe_vectors.shape[1], "distinct ingredients")
print()

# A pantry becomes a vector with the SAME vocabulary. transform, not fit.
pantry_vector = vectorizer.transform([PANTRY])
print("the pantry becomes", pantry_vector.shape)

In [ ]:
# What every ingredient is worth, measured over all 100 recipes.
weights = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())

print("in almost every recipe, so worth almost nothing:")
print(weights.sort_values().head(5).round(3).to_string())
print()
print("in one or two, so decisive:")
for ingredient in ["chickpeas", "spinach", "saffron", "tahini", "sumac"]:
    print(f"{ingredient:12s} {weights[ingredient]:.3f}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# One call scores the pantry against all 100 recipes.
pantry_scores = cosine_similarity(pantry_vector, recipe_vectors)[0]

ranked = pd.DataFrame({"name": recipes["name"], "score": pantry_scores})
ranked = ranked.sort_values("score", ascending=False)

print("PANTRY:", PANTRY)
print()
print(ranked.head(5).round(3).to_string(index=False))
print()
print("Chana saag uses five of your six. Ctrl+F could not have told you that,")
print("because no recipe uses all six and Ctrl+F cannot count partial matches.")


---

## Your turn

Two cells: finish the search, then use it on your own pantry. Work in your team.


In [ ]:
# ================== YOUR TURN 1 ==================
# `search` scores only the FIRST recipe, so it always returns Hummus.
# Score every recipe, then sort by score before taking the top few.
#
# Hint: cosine_similarity(pantry_vector, recipe_vectors) returns one row of 100 scores, so you may not need a loop at all.
#
# Expected: three rows: Chana saag 0.828, Hummus 0.517, Falafel 0.401.
#           While the loop is unfinished you get one row, Hummus at 0.517,
#           because it only ever scores recipe number one.
# =================================================
def search(pantry, how_many=3):
    """Return the how_many recipes that best match a pantry."""
    pantry_vector = vectorizer.transform([pantry])

    found = []
    for row_number in range(1):                # <-- should be every recipe
        score = cosine_similarity(pantry_vector, recipe_vectors[row_number])[0][0]
        found.append({"name": recipes["name"][row_number], "score": score})

    results = pd.DataFrame(found)
    # <-- sort by score, highest first, before taking the top few
    return results.head(how_many)


print(search(PANTRY).round(3).to_string(index=False))

In [ ]:
# ================== YOUR TURN 2 ==================
# Your team's pantry is below. Answer three questions with it.
#
#   a. What are the three best things you can cook tonight?
#   b. You can buy ONE more ingredient. Which of the six candidates gets
#      you the best dish, and which one changes which dish wins?
#   c. Which ingredient already in your pantry is doing the least work?
#
# Expected: a. Arroz con pollo 0.803, Paella 0.751, Fried rice 0.474.
#           b. saffron is the best buy, taking Arroz con pollo to 0.887.
#              chorizo scores lower, 0.829, but hands the win to Paella.
#           c. salt, with the smallest idf of the six at 1.246. It is in 78 of
#              the 100 recipes, so it never separates anything.
# =================================================
MY_PANTRY = "rice onion garlic peas chicken stock salt"
CANDIDATES = ["saffron", "chorizo", "mushrooms", "tomato", "butter", "egg"]

print("a. tonight:")
print(search(MY_PANTRY).round(3).to_string(index=False))

print()
print("b. buy one more:")
for candidate in CANDIDATES:
    best = search(MY_PANTRY + " " + candidate, how_many=1)
    print(f"   +{candidate:11s} {best['name'].iloc[0]:20s} {best['score'].iloc[0]:.3f}")

print()
print("c. what each of your ingredients is worth:")
for ingredient in MY_PANTRY.split():
    print(f"   {ingredient:9s} idf {weights[ingredient]:.3f}")

## Answers

Try each task before reading.

In [ ]:
# TRY IT 1
#   [term_frequencies(d)[VOCAB.index("salt")] for d in DOCS]
#   0.167, 0.143, 0.200, 0.167. NOT the same, because the recipes are 6, 7, 5
#   and 6 words long and tf divides by the length. One salt in a short recipe is
#   a bigger share of it.

# TRY IT 2
#   pd.Series(tf_idf(DOCS[0]), index=VOCAB).sort_values(ascending=False)
#   parsley, 0.2310, on its own at the top: it is the only one of the four
#   recipes that has it. chickpeas and lemon follow at 0.1155, olive and oil at
#   0.0479, and salt is 0.0000 no matter how much of it you use.

# TRY IT 3
#   cosine(tf_idf("olive oil salt"), all_weights[i]) for each i
#   0.233, 0.182, 0.000, 0.321. It ranks them confidently and the ranking is
#   worthless: salt contributes nothing at all, and olive oil is in three of the
#   four. Recipe 3 scores 0.000 only because it happens to have no oil. A search
#   engine always returns an order. Whether the order means anything is a
#   separate question, and it depends on whether your words were rare.

# YOUR TURN 1
#   def search(pantry, how_many=3):
#       pantry_vector = vectorizer.transform([pantry])
#       scores = cosine_similarity(pantry_vector, recipe_vectors)[0]
#       results = pd.DataFrame({"name": recipes["name"], "score": scores})
#       return results.sort_values("score", ascending=False).head(how_many)
#
#   Chana saag 0.828, Hummus 0.517, Falafel 0.401.
#   The loop version works too and is worth writing once:
#       for row_number in range(recipe_vectors.shape[0]):
#           score = cosine_similarity(pantry_vector, recipe_vectors[row_number])[0][0]
#   It is about a hundred times slower: the loop runs in Python and the one-shot
#   version runs in C. Note this is scikit-learn's
#   cosine_similarity, doing over a whole matrix what our four-line cosine() did
#   over one pair.

# YOUR TURN 2
#   a.  Arroz con pollo 0.803, Paella 0.751, Fried rice 0.474.
#
#   b.  saffron    -> Arroz con pollo 0.887   the best score you can buy
#       chorizo    -> Paella          0.829   lower, but it changes the winner
#       butter     -> Arroz con pollo 0.766
#       tomato     -> Arroz con pollo 0.761
#       egg        -> Arroz con pollo 0.733
#       mushrooms  -> Arroz con pollo 0.703
#
#       Saffron is in 3 recipes out of 100 and chorizo in 3, so both are worth a
#       lot. Every other candidate LOWERS the score: adding an ingredient no good
#       recipe has lengthens your pantry vector without matching anything, and
#       cosine divides by that length. A longer shopping list is not a better
#       query.
#
#   c.  salt, idf 1.246, in 78 of the 100 recipes. Then onion 1.723 and garlic
#       1.854. Your rare ingredients are peas 4.229 and rice 3.536, and they are
#       what put paella and arroz con pollo at the top.

# The three things worth carrying out of today:
#   1. Ctrl+F finds a string. tf-idf RANKS partial matches, which is a different
#      question and the one you usually actually have.
#   2. A word is worth what it narrows down. Salt is in everything, so it is
#      worth nothing, and the arithmetic works that out on its own.
#   3. Cosine compares direction, not size, so a long recipe and a short one can
#      still be about the same dish.